In [ ]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from collections import Counter
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("ATTENTION-BASED ENGLISH-TAMIL TRANSLATOR")
print("=" * 60)

print("\nDevice:", device)
FILE_PATH = "corpus.bcn.train 166k.csv"

df = pd.read_csv(
    FILE_PATH,
    encoding="utf-8"
)

print("\n" + "=" * 60)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 60)

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())
english_column = "en"
tamil_column = "ta"

print("\nEnglish Column:")
print(english_column)

print("\nTamil Column:")
print(tamil_column)
df = df[
    [
        english_column,
        tamil_column
    ]
].copy()


df.columns = [
    "english",
    "tamil"
]
df = df.dropna(
    subset=[
        "english",
        "tamil"
    ]
).reset_index(
    drop=True
)


df["english"] = df[
    "english"
].astype(str)

df["tamil"] = df[
    "tamil"
].astype(str)

df = df[
    (df["english"].str.strip() != "")
    &
    (df["tamil"].str.strip() != "")
].reset_index(
    drop=True
)


print("\n" + "=" * 60)
print("CLEANED DATASET")
print("=" * 60)

print("\nShape:")
print(df.shape)

print("\nFirst 5 Sentence Pairs:")
print(df.head())
MAX_SAMPLES = 20000

if MAX_SAMPLES is not None:

    df = df.head(
        MAX_SAMPLES
    ).copy()


print("\nTraining Samples:")
print(len(df))
def tokenize_english(sentence):

    return str(
        sentence
    ).lower().split()


def tokenize_tamil(sentence):

    return str(
        sentence
    ).split()

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
MAX_ENGLISH_VOCAB = 10000
MAX_TAMIL_VOCAB = 15000

MAX_SOURCE_LENGTH = 30
MAX_TARGET_LENGTH = 30

english_counter = Counter()

for sentence in df["english"]:

    tokens = tokenize_english(
        sentence
    )

    english_counter.update(
        tokens
    )


english_vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1,
    SOS_TOKEN: 2,
    EOS_TOKEN: 3
}


for word, count in english_counter.most_common(
    MAX_ENGLISH_VOCAB
):

    if word not in english_vocab:

        english_vocab[word] = len(
            english_vocab
        )
tamil_counter = Counter()

for sentence in df["tamil"]:

    tokens = tokenize_tamil(
        sentence
    )

    tamil_counter.update(
        tokens
    )


tamil_vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1,
    SOS_TOKEN: 2,
    EOS_TOKEN: 3
}

for word, count in tamil_counter.most_common(
    MAX_TAMIL_VOCAB
):

    if word not in tamil_vocab:

        tamil_vocab[word] = len(
            tamil_vocab
        )
english_index_to_word = {
    index: word
    for word, index
    in english_vocab.items()
}
tamil_index_to_word = {
    index: word
    for word, index
    in tamil_vocab.items()
}
print("\n" + "=" * 60)
print("VOCABULARY")
print("=" * 60)

print(
    "English Vocabulary:",
    len(english_vocab)
)

print(
    "Tamil Vocabulary:",
    len(tamil_vocab)
)
def english_to_indices(
    sentence
):
    tokens = tokenize_english(
        sentence
    )
    tokens = tokens[
        :MAX_SOURCE_LENGTH - 2
    ]
    sequence = [
        english_vocab[SOS_TOKEN]
    ]
    for token in tokens:

        sequence.append(
            english_vocab.get(
                token,
                english_vocab[UNK_TOKEN]
            )
        )
    sequence.append(
        english_vocab[EOS_TOKEN]
    )
    return sequence
def tamil_to_indices(
    sentence
):

    tokens = tokenize_tamil(
        sentence
    )

    tokens = tokens[
        :MAX_TARGET_LENGTH - 2
    ]
    sequence = [
        tamil_vocab[SOS_TOKEN]
    ]
    for token in tokens:
        sequence.append(
            tamil_vocab.get(
                token,
                tamil_vocab[UNK_TOKEN]
            )
        )
    sequence.append(
        tamil_vocab[EOS_TOKEN]
    )
    return sequence
def pad_english(
    sequence
):
    sequence = sequence[
        :MAX_SOURCE_LENGTH
    ]
    while len(sequence) < MAX_SOURCE_LENGTH:

        sequence.append(
            english_vocab[PAD_TOKEN]
        )
    return sequence
def pad_tamil(
    sequence
):
    sequence = sequence[
        :MAX_TARGET_LENGTH
    ]
    while len(sequence) < MAX_TARGET_LENGTH:

        sequence.append(
            tamil_vocab[PAD_TOKEN]
        )
    return sequence

class TranslationDataset(
    Dataset
):

    def __init__(
        self,
        dataframe
    ):

        self.english = dataframe[
            "english"
        ].tolist()

        self.tamil = dataframe[
            "tamil"
        ].tolist()
    def __len__(
        self
    ):
        return len(
            self.english
        )
    def __getitem__(
        self,
        index
    ):
        source = english_to_indices(
            self.english[index]
        )

        target = tamil_to_indices(
            self.tamil[index]
        )
        source = pad_english(
            source
        )
        target = pad_tamil(
            target
        )
        return (
            torch.tensor(
                source,
                dtype=torch.long
            ),

            torch.tensor(
                target,
                dtype=torch.long
            )
        )
split_index = int(
    len(df) * 0.9
)

train_df = df[
    :split_index
].reset_index(
    drop=True
)

valid_df = df[
    split_index:
].reset_index(
    drop=True
)

print("\n" + "=" * 60)
print("DATA SPLIT")
print("=" * 60)

print(
    "Training Samples:",
    len(train_df)
)

print(
    "Validation Samples:",
    len(valid_df)
)
train_dataset = TranslationDataset(
    train_df
)

valid_dataset = TranslationDataset(
    valid_df
)

BATCH_SIZE = 64
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)
class Encoder(
    nn.Module
):

    def __init__(
        self,
        input_size,
        embedding_dim,
        hidden_dim
    ):
        super().__init__()
        self.embedding = nn.Embedding(
            input_size,
            embedding_dim,
            padding_idx=0
        )
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.hidden_fc = nn.Linear(
            hidden_dim * 2,
            hidden_dim
        )
        self.cell_fc = nn.Linear(
            hidden_dim * 2,
            hidden_dim
        )
    def forward(
        self,
        source
    ):
        embedded = self.embedding(
            source
        )

        outputs, (
            hidden,
            cell
        ) = self.lstm(
            embedded
        )

        hidden = torch.cat(
            (
                hidden[-2],
                hidden[-1]
            ),
            dim=1
        )
        cell = torch.cat(
            (
                cell[-2],
                cell[-1]
            ),
            dim=1
        )


        hidden = torch.tanh(
            self.hidden_fc(
                hidden
            )
        )
        cell = torch.tanh(
            self.cell_fc(
                cell
            )
        )
        hidden = hidden.unsqueeze(
            0
        )
        cell = cell.unsqueeze(
            0
        )
        return (
            outputs,
            hidden,
            cell
        )


# ============================================================
# 21. ATTENTION
# ============================================================

class Attention(
    nn.Module
):

    def __init__(
        self,
        encoder_dim,
        decoder_dim
    ):

        super().__init__()


        self.attention = nn.Linear(
            encoder_dim + decoder_dim,
            1
        )


    def forward(
        self,
        hidden,
        encoder_outputs
    ):

        source_length = (
            encoder_outputs.size(1)
        )


        hidden = hidden.unsqueeze(
            1
        )


        hidden = hidden.repeat(
            1,
            source_length,
            1
        )


        energy = torch.tanh(
            self.attention(
                torch.cat(
                    (
                        hidden,
                        encoder_outputs
                    ),
                    dim=2
                )
            )
        )


        attention_weights = torch.softmax(
            energy.squeeze(2),
            dim=1
        )


        return attention_weights


# ============================================================
# 22. DECODER
# ============================================================

class Decoder(
    nn.Module
):

    def __init__(
        self,
        output_size,
        embedding_dim,
        encoder_dim,
        decoder_dim
    ):

        super().__init__()


        self.output_size = output_size


        self.embedding = nn.Embedding(
            output_size,
            embedding_dim,
            padding_idx=0
        )


        self.attention = Attention(
            encoder_dim,
            decoder_dim
        )


        self.lstm = nn.LSTM(
            embedding_dim + encoder_dim,
            decoder_dim,
            batch_first=True
        )


        self.fc = nn.Linear(
            decoder_dim
            + encoder_dim
            + embedding_dim,
            output_size
        )


    def forward(
        self,
        input_token,
        hidden,
        cell,
        encoder_outputs
    ):

        input_token = input_token.unsqueeze(
            1
        )


        embedded = self.embedding(
            input_token
        )


        attention_weights = self.attention(
            hidden[-1],
            encoder_outputs
        )


        attention_weights = (
            attention_weights.unsqueeze(1)
        )


        context = torch.bmm(
            attention_weights,
            encoder_outputs
        )


        rnn_input = torch.cat(
            (
                embedded,
                context
            ),
            dim=2
        )
        output, (
            hidden,
            cell
        ) = self.lstm(
            rnn_input,
            (hidden, cell)
        )
        output = output.squeeze(
            1
        )
        context = context.squeeze(
            1
        )
        embedded = embedded.squeeze(
            1
        )
        prediction = self.fc(
            torch.cat(
                (
                    output,
                    context,
                    embedded
                ),
                dim=1
            )
        )


        return (
            prediction,
            hidden,
            cell,
            attention_weights.squeeze(1)
        )
class Seq2Seq(
    nn.Module
):

    def __init__(
        self,
        encoder,
        decoder,
        device
    ):

        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    def forward(
        self,
        source,
        target,
        teacher_forcing_ratio=0.5
    ):

        batch_size = source.size(0)
        target_length = target.size(1)
        target_vocab_size = (
            self.decoder.output_size
        )
        outputs = torch.zeros(
            batch_size,
            target_length,
            target_vocab_size,
            device=self.device
        )
        encoder_outputs, hidden, cell = (
            self.encoder(
                source
            )
        )
        input_token = target[
            :,
            0
        ]
        for t in range(
            1,
            target_length
        ):

            output, hidden, cell, _ = (
                self.decoder(
                    input_token,
                    hidden,
                    cell,
                    encoder_outputs
                )
            )
            outputs[
                :,
                t,
                :
            ] = output
            best_guess = output.argmax(
                1
            )
            teacher_force = (
                np.random.random()
                <
                teacher_forcing_ratio
            )
            if teacher_force:
                input_token = target[
                    :,
                    t
                ]
            else:
                input_token = best_guess
        return outputs
INPUT_SIZE = len(
    english_vocab
)

OUTPUT_SIZE = len(
    tamil_vocab
)

ENCODER_EMBEDDING = 128
DECODER_EMBEDDING = 128
ENCODER_HIDDEN = 256
DECODER_HIDDEN = 256

encoder = Encoder(
    input_size=INPUT_SIZE,
    embedding_dim=ENCODER_EMBEDDING,
    hidden_dim=ENCODER_HIDDEN
)
decoder = Decoder(
    output_size=OUTPUT_SIZE,
    embedding_dim=DECODER_EMBEDDING,
    encoder_dim=ENCODER_HIDDEN * 2,
    decoder_dim=DECODER_HIDDEN
)
model = Seq2Seq(
    encoder,
    decoder,
    device
).to(device)


print("\n" + "=" * 60)
print("MODEL")
print("=" * 60)

print(model)
criterion = nn.CrossEntropyLoss(
    ignore_index=tamil_vocab[PAD_TOKEN]
)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
def train_model(
    model,
    loader
):
    model.train()
    total_loss = 0
    for source, target in loader:
        source = source.to(
            device
        )

        target = target.to(
            device
        )
        optimizer.zero_grad()
        output = model(
            source,
            target,
            teacher_forcing_ratio=0.5
        )
        output_dim = output.size(
            -1
        )
        output = output[
            :,
            1:,
            :
        ].contiguous().view(
            -1,
            output_dim
        )
        target = target[
            :,
            1:
        ].contiguous().view(
            -1
        )
        loss = criterion(
            output,
            target
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1
        )
        optimizer.step()
        total_loss += loss.item()
    return (
        total_loss
        /
        len(loader)
    )
def evaluate_model(
    model,
    loader
):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for source, target in loader:

            source = source.to(
                device
            )
            target = target.to(
                device
            )
            output = model(
                source,
                target,
                teacher_forcing_ratio=0
            )
            output_dim = output.size(
                -1
            )
            output = output[
                :,
                1:,
                :
            ].contiguous().view(
                -1,
                output_dim
            )
            target = target[
                :,
                1:
            ].contiguous().view(
                -1
            )
            loss = criterion(
                output,
                target
            )
            total_loss += loss.item()
    return (
        total_loss
        /
        len(loader)
    )
EPOCHS = 5
print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)
for epoch in range(
    EPOCHS
):
    train_loss = train_model(
        model,
        train_loader
    )
    validation_loss = evaluate_model(
        model,
        valid_loader
    )
    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Validation Loss: {validation_loss:.4f}"
    )
def translate_sentence(
    sentence
):
    model.eval()
    source_indices = english_to_indices(
        sentence
    )
    source_indices = pad_english(
        source_indices
    )
    source_tensor = torch.tensor(
        source_indices,
        dtype=torch.long
    ).unsqueeze(
        0
    ).to(device)
    with torch.no_grad():
        encoder_outputs, hidden, cell = (
            model.encoder(
                source_tensor
            )
        )
    input_token = torch.tensor(
        [
            tamil_vocab[SOS_TOKEN]
        ],
        dtype=torch.long
    ).to(device)
    translated_words = []
    for _ in range(
        MAX_TARGET_LENGTH
    ):
        with torch.no_grad():

            output, hidden, cell, attention = (
                model.decoder(
                    input_token,
                    hidden,
                    cell,
                    encoder_outputs
                )
            )
        predicted_token = output.argmax(
            1
        ).item()

        if predicted_token == tamil_vocab[
            EOS_TOKEN
        ]:
            break
        if predicted_token not in [
            tamil_vocab[PAD_TOKEN],
            tamil_vocab[SOS_TOKEN],
            tamil_vocab[UNK_TOKEN]
        ]:
            word = tamil_index_to_word.get(
                predicted_token,
                ""
            )
            if word != "":

                translated_words.append(
                    word
                )
        input_token = torch.tensor(
            [predicted_token],
            dtype=torch.long
        ).to(device)
    return " ".join(
        translated_words
    )
print("\n" + "=" * 60)
print("TEST TRANSLATIONS")
print("=" * 60)
for i in range(
    min(5, len(valid_df))
):
    english_sentence = valid_df[
        "english"
    ].iloc[i]
    actual_tamil = valid_df[
        "tamil"
    ].iloc[i]
    predicted_tamil = (
        translate_sentence(
            english_sentence
        )
    )
    print("\nEnglish:")
    print(english_sentence)

    print("\nActual Tamil:")
    print(actual_tamil)

    print("\nPredicted Tamil:")
    print(predicted_tamil)
    print("-" * 60)
print("\n" + "=" * 60)
print("REAL-TIME ENGLISH TO TAMIL TRANSLATOR")
print("=" * 60)

while True:
    english_input = input(
        "\nEnter English sentence "
        "(type 'exit' to stop): "
    )
    if english_input.lower().strip() == "exit":
        break
    tamil_output = translate_sentence(
        english_input
    )
    print("\nTamil Translation:")
    print(tamil_output)
torch.save(
    model.state_dict(),
    "english_tamil_attention_model.pth"
)
np.save(
    "english_vocab.npy",
    english_vocab,
    allow_pickle=True
)
np.save(
    "tamil_vocab.npy",
    tamil_vocab,
    allow_pickle=True
)
print("\n" + "=" * 60)
print("MODEL SAVED SUCCESSFULLY")
print("=" * 60)

print(
    "english_tamil_attention_model.pth"
)

print(
    "english_vocab.npy"
)

print(
    "tamil_vocab.npy"
)

print("\nPROGRAM COMPLETED")
print("=" * 60)

ATTENTION-BASED ENGLISH-TAMIL TRANSLATOR

Device: cpu

DATASET LOADED SUCCESSFULLY

Dataset Shape:
(166871, 3)

Column Names:
['Unnamed: 0', 'en', 'ta']

First 5 Rows:
   Unnamed: 0                                                 en  \
0           0  MMA vice president Qazi Hussain Ahmad declared...   
1           1  Information has surfaced in recent years sugge...   
2           2  And Azor begat Sadoc; and Sadoc begat Achim; a...   
3           3  She says she knows what is going on, but can d...   
4           4  And be it indeed that I have erred, my error r...   

                                                  ta  
0  MMA கட்சியின் துணைத்தலைவர் க்வாஸி ஹுசேன் அகமத்...  
1  சமீபகாலத்தில் சில தகவல்கள் யூலியஸ் ரோசன்பேர்க...  
2  ஆசோர் சாதோக்கைப் பெற்றான்; சாதோக்கு ஆகீமைப் பெ...  
3  என்ன நடக்கிறது என்பது தமக்கு தெரியும் என்றும் ...  
4  நான் தப்பிநடந்தது மெய்யானாலும், என் தப்பிதம் எ...  

English Column:
en

Tamil Column:
ta

CLEANED DATASET

Shape:
(166871, 2)

First 5 Sentence 